# Part 11 — Scores, shortcuts, and 20-step sampling

_Rigorous Courses · Diffusion Models — Part 11 of 12_

**Read the trained network as a compass pointing uphill toward realistic data, and 200 sampling steps collapse to 20**

In this notebook you retrain part 10's DDPM in seconds, then re-read it: the noise guess, negated and rescaled, is the **score** — the uphill direction of the log density — and Tweedie's formula proves it on a toy where every quantity has a closed form. From the score you build the denoised estimate $\hat{x}_0$ and the deterministic DDIM sampler, race 20-step sub-schedules against naive skipping, and close with a bit-for-bit determinism check that turns starting noise into a latent code.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab; torch is preinstalled there too.
import math

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

rng = np.random.default_rng(0)
torch.manual_seed(0)

print(f"torch version: {torch.__version__} (CPU is all we need)")

## Part 10's machine, rebuilt

Everything in this part re-reads a network we already have, so first we rebuild part 10's setup — the eight-mode ring, the $T = 200$ linear schedule, the sinusoidal time embedding, the 21,250-parameter MLP — and retrain it. Nothing new here: this is part 10's code with the commentary stripped, condensed into two cells.

### Step 1 — Rebuild the ring, the schedule, and the network

One cell holds what part 10 built over six: the dataset (8 Gaussian clusters of spread 0.15 on a circle of radius 4), the schedule arrays `betas`, `alphas`, `abar`, the embedding, and the MLP. The asserts pin the load-bearing facts: the data shape, that $\bar\alpha_t$ strictly falls, and the exact parameter count of 21,250.

In [ ]:
n = 4096
T = 200

mode = rng.integers(0, 8, size=n)
angle = 2 * np.pi * mode / 8
centers = 4.0 * np.stack([np.cos(angle), np.sin(angle)], axis=1)
x0_np = centers + 0.15 * rng.standard_normal((n, 2))
data = torch.tensor(x0_np, dtype=torch.float32)

betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
abar = torch.cumprod(alphas, dim=0)
sqrt_abar = torch.sqrt(abar)
sqrt_1m_abar = torch.sqrt(1.0 - abar)

emb_dim = 32
half = emb_dim // 2
freqs = torch.exp(-math.log(10000.0) * torch.arange(half) / (half - 1))


def time_embedding(t):
    args = t[:, None].float() * freqs[None, :]
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    return emb


class EpsMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 + emb_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x, t):
        emb = time_embedding(t)
        h = torch.cat([x, emb], dim=1)
        return self.net(h)


model = EpsMLP()
n_params = sum(p.numel() for p in model.parameters())

assert data.shape == (4096, 2)
assert torch.all(abar[1:] < abar[:-1])
assert n_params == 21250

print(f"data shape: {tuple(data.shape)}   mean radius: {np.linalg.norm(x0_np, axis=1).mean():.3f} (target 4.0)")
print(f"abar_1 = {abar[0].item():.4f}   abar_T = {abar[-1].item():.4f}")
print(f"parameter count: {n_params} (part 10's hand count)")

### Step 2 — Retrain for 2,000 steps

Algorithm 1 with minibatches, exactly as in part 10: sample clean points, sample a timestep per point, sample fresh noise, form $x_t$ with the one-line sampler $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$, and take one Adam step on the mean squared error. The loss should start near 1.0 (an untrained network guesses about 0, and the noise has variance 1 per coordinate) and fall to roughly half — the assert checks the drop.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

for step in range(2000):
    idx = torch.randint(0, n, (256,))
    x0 = data[idx]
    t = torch.randint(1, T + 1, (256,))
    eps = torch.randn(256, 2)
    xt = sqrt_abar[t - 1][:, None] * x0 + sqrt_1m_abar[t - 1][:, None] * eps
    eps_pred = model(xt, t)
    loss = ((eps - eps_pred) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 250 == 0 or step == 1999:
        print(f"step {step:4d}   batch loss = {loss.item():.3f}")

first_loss = losses[0]
late_loss = float(np.mean(losses[-100:]))

assert late_loss < first_loss * 0.6

print(f"first batch loss: {first_loss:.3f}   mean of last 100: {late_loss:.3f} — trained")

## The score: which way is uphill?

The lesson derived the score of a Gaussian in seven one-op steps:

$$\nabla_x \log \mathcal{N}(x;\, \mu, \sigma^2) = \frac{\mu - x}{\sigma^2}$$

— a pull back toward the center that grows with distance and weakens when the bell is wide. Slopes are checkable without any calculus: a **finite difference** $\big(\log p(x + h) - \log p(x - h)\big) / (2h)$ with a tiny $h$ measures the slope directly from the density. If the formula is right, the two must agree everywhere.

### Step 3 — Check the Gaussian score against a finite-difference slope

The lesson's worked example says: for $\mathcal{N}(1, 4)$ the score at $x = 3$ is $(1 - 3)/4 = -0.5$, at the center $x = 1$ it is $0$ (hilltops are flat), and at $x = -7$ it is $+2$ (four times the distance, twice the pull... divided by the variance 4). We measure all three numerically, then sweep a whole grid.

In [ ]:
mu = 1.0
sigma2 = 4.0


def log_pdf(x):
    return -0.5 * (x - mu) ** 2 / sigma2 - 0.5 * np.log(2 * np.pi * sigma2)


def score_formula(x):
    return (mu - x) / sigma2


h = 1e-5
x_checks = np.array([3.0, 1.0, -7.0])
fd = (log_pdf(x_checks + h) - log_pdf(x_checks - h)) / (2 * h)

grid = np.linspace(-9.0, 11.0, 401)
fd_grid = (log_pdf(grid + h) - log_pdf(grid - h)) / (2 * h)
worst = np.max(np.abs(fd_grid - score_formula(grid)))

assert abs(fd[0] - (-0.5)) < 1e-6
assert abs(fd[1]) < 1e-6
assert abs(fd[2] - 2.0) < 1e-6
assert worst < 1e-6

print(f"score at x =  3: formula {score_formula(3.0):+.4f}   finite difference {fd[0]:+.4f}")
print(f"score at x =  1: formula {score_formula(1.0):+.4f}   finite difference {fd[1]:+.4f}")
print(f"score at x = -7: formula {score_formula(-7.0):+.4f}   finite difference {fd[2]:+.4f}")
print(f"worst gap over a 401-point grid: {worst:.2e}")

### Step 4 — Picture the pull toward the center

Left: the bell of $\mathcal{N}(1, 4)$. Right: its score — a straight line, zero exactly at the center, negative to the right of it (uphill points left, back home) and positive to the left. The three dots mark the worked example's points.

In [ ]:
pdf = np.exp(log_pdf(grid))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(grid, pdf, color="#4ea1ff")
ax1.axvline(mu, color="#ff7b72", ls="--", lw=1)
ax1.set_xlabel("x")
ax1.set_ylabel("density")
ax1.set_title("N(1, 4): the bell")
ax2.plot(grid, score_formula(grid), color="#4ea1ff")
ax2.axhline(0.0, color="gray", lw=0.5)
ax2.plot([3.0, 1.0, -7.0], [-0.5, 0.0, 2.0], "o", color="#ff7b72")
ax2.set_xlabel("x")
ax2.set_ylabel("score")
ax2.set_title("its score: (mu - x) / sigma^2")
plt.tight_layout()
plt.show()

## Tweedie's formula: the network is secretly a compass

For the two-point toy from part 9 ($x_0 = +2$ or $-2$, equal odds) the noised density at level $t$ is a mixture of two shrunken-center bells, and the lesson derived Tweedie's formula plus the bridge to the ideal noise guess:

$$\nabla_{x_t} \log q(x_t) \;=\; \frac{\sqrt{\bar\alpha_t}\;\mathbb{E}[x_0 \mid x_t] - x_t}{1-\bar\alpha_t} \;=\; -\,\frac{\epsilon_\theta^*(x_t, t)}{\sqrt{1-\bar\alpha_t}}$$

Every quantity here has a closed form (part 9's tanh posterior mean), and the score can *also* be measured by a finite difference on the raw mixture density — no posterior, no derivation. Three independent routes to one number: they must all agree.

### Step 5 — Build the mixture density and the exact posterior mean

Part 9's two functions, verbatim: the tanh formula for $\mathbb{E}[x_0 \mid x_t]$ and the best noise guess built from it — plus the raw mixture density $q$ itself. (We take the schedule in float64 here: this section checks *exact algebra*, so we want the arithmetic itself at full precision.) First probe: at $x_t = 0$ the evidence for $+2$ and $-2$ is perfectly balanced, so the posterior mean and the best noise guess must both be zero.

In [ ]:
abar_np = abar.double().numpy()


def gauss_pdf(x, mean, var):
    return np.exp(-0.5 * (x - mean) ** 2 / var) / np.sqrt(2 * np.pi * var)


def q_mix(x, ab_t):
    a = np.sqrt(ab_t)
    v = 1.0 - ab_t
    return 0.5 * gauss_pdf(x, 2.0 * a, v) + 0.5 * gauss_pdf(x, -2.0 * a, v)


def x0_best_guess(x, ab_t):
    a = np.sqrt(ab_t)
    v = 1.0 - ab_t
    return 2.0 * np.tanh(2.0 * a * x / v)


def eps_best_guess(x, ab_t):
    a = np.sqrt(ab_t)
    v = 1.0 - ab_t
    return (x - a * x0_best_guess(x, ab_t)) / np.sqrt(v)


probe_mean = x0_best_guess(0.0, abar_np[99])
probe_eps = eps_best_guess(0.0, abar_np[99])

assert abs(probe_mean) < 1e-12
assert abs(probe_eps) < 1e-12

print(f"at x_t = 0, t = 100:  E[x0 | x_t] = {probe_mean:.6f}   eps* = {probe_eps:.6f}   (symmetry says both must be 0)")

### Step 6 — Check the bridge three ways across the whole grid

At three noise levels we compute the score by (1) Tweedie's formula, (2) the bridge $-\epsilon^*/\sqrt{1-\bar\alpha_t}$, and (3) a finite-difference slope of $\log q$ from the raw mixture density. The lesson's sanity check promises agreement better than $10^{-3}$; we get far better. And watch the curves steepen as $t$ falls — near-clean densities are cliff-steep around the data, exactly the "scale across $t$" behavior the lesson describes.

In [ ]:
t_show = [180, 100, 20]
x_grid = np.linspace(-3.5, 3.5, 281)
h_fd = 1e-6

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
worst_bridge = 0.0
worst_fd = 0.0
for ax, t_step in zip(axes, t_show):
    ab_t = abar_np[t_step - 1]
    s = np.sqrt(1.0 - ab_t)
    score_tweedie = (np.sqrt(ab_t) * x0_best_guess(x_grid, ab_t) - x_grid) / (1.0 - ab_t)
    score_bridge = -eps_best_guess(x_grid, ab_t) / s
    score_fd = (np.log(q_mix(x_grid + h_fd, ab_t)) - np.log(q_mix(x_grid - h_fd, ab_t))) / (2 * h_fd)
    worst_bridge = max(worst_bridge, np.max(np.abs(score_tweedie - score_bridge)))
    worst_fd = max(worst_fd, np.max(np.abs(score_tweedie - score_fd)))
    ax.plot(x_grid, score_tweedie, color="#4ea1ff", label="Tweedie formula")
    ax.plot(x_grid[::14], score_fd[::14], "o", ms=3, color="#ff7b72", label="finite diff of log q")
    ax.set_title(f"t = {t_step}")
    ax.set_xlabel("x_t")
axes[0].set_ylabel("score")
axes[0].legend()
plt.suptitle("One score, three routes: they agree at every point and every level")
plt.tight_layout()
plt.show()

assert worst_bridge < 1e-10
assert worst_fd < 1e-3

print(f"worst Tweedie-vs-bridge gap:            {worst_bridge:.2e} (exact algebra)")
print(f"worst Tweedie-vs-finite-difference gap: {worst_fd:.2e} (well under the promised 1e-3)")

### Step 7 — Verify the lesson's hand example to four decimals

Running schedule at $t = 3$: $\bar\alpha_3 = 0.504$, observe $x_3 = 0.3$. The lesson computed by hand: $\mathbb{E}[x_0 \mid x_3] = 1.3913$, score $= 1.3865$, best noise guess $= -0.9765$ — and the bridge check reproduced the score from the noise guess. Same numbers, by code.

In [ ]:
ab3 = 0.504
x_obs = 0.3
a3 = np.sqrt(ab3)
s3 = np.sqrt(1.0 - ab3)

post_mean = x0_best_guess(x_obs, ab3)
score_hand = (a3 * post_mean - x_obs) / (1.0 - ab3)
eps_hand = eps_best_guess(x_obs, ab3)
bridge_hand = -eps_hand / s3

assert abs(post_mean - 1.3913) < 5e-4
assert abs(score_hand - 1.3865) < 5e-4
assert abs(eps_hand - (-0.9765)) < 5e-4
assert abs(bridge_hand - score_hand) < 1e-12

print(f"E[x0 | x3 = 0.3] = {post_mean:.4f}   (lesson: 1.3913)")
print(f"score via Tweedie = {score_hand:.4f}   (lesson: 1.3865)")
print(f"best noise guess  = {eps_hand:.4f}   (lesson: -0.9765)")
print(f"bridge -eps*/s    = {bridge_hand:.4f}   — the same number, as derived")

### Step 8 — Draw the compass: the learned score field at three noise levels

Now the real network. At each point of a grid we ask the trained model for its noise guess and convert it with the bridge: score $= -\epsilon_\theta(x_t, t)/\sqrt{1-\bar\alpha_t}$. Read the panels left to right ($t = 150, 75, 20$): the arrows always point toward the ring (faint red dots), and the field sharpens dramatically as the noise level falls — the $1/\sqrt{1-\bar\alpha_t}$ scale grows, and the landscape turns from gentle hill into cliff. Arrows are drawn at unit length so directions stay readable; color carries the true magnitude.

In [ ]:
gv = np.linspace(-6.0, 6.0, 17)
gx, gy = np.meshgrid(gv, gv)
grid_pts = torch.tensor(np.stack([gx.ravel(), gy.ravel()], axis=1), dtype=torch.float32)

t_field = [150, 75, 20]
mean_mags = []

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for ax, t_step in zip(axes, t_field):
    tvec = torch.full((grid_pts.shape[0],), t_step)
    with torch.no_grad():
        eps_pred = model(grid_pts, tvec)
    score_field = (-eps_pred / sqrt_1m_abar[t_step - 1]).numpy()
    mag = np.linalg.norm(score_field, axis=1)
    unit = score_field / (mag[:, None] + 1e-12)
    mean_mags.append(mag.mean())
    ax.quiver(gx, gy, unit[:, 0].reshape(gx.shape), unit[:, 1].reshape(gx.shape), mag.reshape(gx.shape), cmap="viridis", scale=28)
    ax.scatter(x0_np[::16, 0], x0_np[::16, 1], s=2, alpha=0.25, color="#ff7b72")
    ax.set_title(f"t = {t_step}   mean |score| = {mag.mean():.1f}")
    ax.set_xlabel("first coordinate")
axes[0].set_ylabel("second coordinate")
plt.suptitle("The learned compass: -eps_theta / sqrt(1 - abar_t) points uphill, toward the ring")
plt.tight_layout()
plt.show()

assert mean_mags[2] > mean_mags[1] > mean_mags[0]

print(f"mean score magnitude at t = 150, 75, 20: {mean_mags[0]:.2f}, {mean_mags[1]:.2f}, {mean_mags[2]:.2f} — sharpening as t falls")

## The denoised estimate: a clean guess at every moment

The bridge's second gift, the reusable primitive:

$$\hat{x}_0(x_t, t) \;=\; \frac{x_t - \sqrt{1-\bar\alpha_t}\;\epsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}}$$

— take the noisy point, subtract the guessed noise scaled the way noise was added, undo the shrinking. The lesson proved in three operations that with a perfect guesser this equals $\mathbb{E}[x_0 \mid x_t]$, the posterior mean of the clean point. We verify that identity exactly on the toy, then watch the real network's clean guess sharpen as $t$ falls.

### Step 9 — On the toy, the primitive IS the posterior mean

Feed the exact noise guess into the $\hat{x}_0$ formula and compare against the tanh posterior mean directly, across the whole grid and a spread of timesteps. The proof was three lines of exact algebra, so the two must agree to machine precision — and the lesson's hand value $\hat{x}_0 = 1.3913$ must drop out at the Tweedie example's numbers.

In [ ]:
def xhat0_toy(x, ab_t):
    a = np.sqrt(ab_t)
    s = np.sqrt(1.0 - ab_t)
    return (x - s * eps_best_guess(x, ab_t)) / a


worst_gap = 0.0
for t_step in [3, 20, 100, 180]:
    ab_t = abar_np[t_step - 1]
    gap = np.max(np.abs(xhat0_toy(x_grid, ab_t) - x0_best_guess(x_grid, ab_t)))
    worst_gap = max(worst_gap, gap)

xhat_hand = xhat0_toy(x_obs, ab3)

assert worst_gap < 1e-12
assert abs(xhat_hand - post_mean) < 1e-12

print(f"worst | xhat0 - E[x0 | x_t] | over grid and timesteps: {worst_gap:.2e}")
print(f"hand example: xhat0 = {xhat_hand:.4f}   (lesson: 1.3913 — the posterior mean, recovered)")

### Step 10 — The real network's clean guess: blurry committee, then sharp answer

Take 1,000 real ring points, noise them to level $t$ with the one-line sampler, and ask the network for $\hat{x}_0$. At $t = 150$ the guess is a blurry committee average of everything the point might become — pulled toward the middle, mean radius well under 4. At $t = 20$ the point's identity is nearly settled and $\hat{x}_0$ sits on the ring.

In [ ]:
def xhat0_model(x, t_step):
    tvec = torch.full((x.shape[0],), t_step)
    with torch.no_grad():
        eps_pred = model(x, tvec)
    return (x - sqrt_1m_abar[t_step - 1] * eps_pred) / sqrt_abar[t_step - 1]


x0_batch = data[:1000]
radii = {}

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, t_step in zip(axes, [150, 20]):
    eps_batch = torch.randn(1000, 2)
    xt_batch = sqrt_abar[t_step - 1] * x0_batch + sqrt_1m_abar[t_step - 1] * eps_batch
    xhat_batch = xhat0_model(xt_batch, t_step)
    radii[t_step] = xhat_batch.norm(dim=1).mean().item()
    ax.scatter(xhat_batch[:, 0].numpy(), xhat_batch[:, 1].numpy(), s=3, alpha=0.4, color="#4ea1ff")
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_title(f"xhat0 at t = {t_step}   mean radius {radii[t_step]:.2f}")
    ax.set_xlabel("first coordinate")
axes[0].set_ylabel("second coordinate")
plt.suptitle("The denoised estimate sharpens as t falls (data radius: 4.0)")
plt.tight_layout()
plt.show()

assert radii[20] > radii[150]
assert abs(radii[20] - 4.0) / 4.0 < 0.15

print(f"mean radius of xhat0:  t = 150 -> {radii[150]:.2f} (committee average, pulled inward)   t = 20 -> {radii[20]:.2f} (on the ring)")

## DDIM: jump to the guess, re-noise deterministically

The whole sampler is one update, applied from $t = T$ down:

$$x_{t-1} \;=\; \sqrt{\bar\alpha_{t-1}}\;\hat{x}_0(x_t, t) \;+\; \sqrt{1-\bar\alpha_{t-1}}\;\epsilon_\theta(x_t, t)$$

— part 6's one-line sampler with hats on: jump all the way to the current clean guess, then re-noise it out to level $t-1$, deterministically, reusing the believed noise direction. No $z$, no randomness, anywhere. And because the update mentions only $\bar\alpha$ at the two endpoints of the jump, the target level does not have to be adjacent — that observation is the next section's payoff.

### Step 11 — One DDIM jump by hand, then in code

The lesson's worked example continues the Tweedie numbers: at $t = 3$ (running schedule) we hold $x_3 = 0.3$, and the exact predictor gave $\epsilon^* = -0.9765$ and $\hat{x}_0 = 1.3913$. The jump to level 2 ($\bar\alpha_2 = 0.72$) lands at $x_2 = 0.6638$ — pulled toward the $+2$ branch the posterior favors, with no coin flipped. Rerunning the update gives the identical number.

In [ ]:
ab2 = 0.72

x2_ddim = np.sqrt(ab2) * xhat_hand + np.sqrt(1.0 - ab2) * eps_hand
x2_again = np.sqrt(ab2) * xhat_hand + np.sqrt(1.0 - ab2) * eps_hand

assert abs(x2_ddim - 0.6638) < 5e-4
assert x2_again == x2_ddim

print(f"DDIM jump: x3 = {x_obs}  ->  x2 = {x2_ddim:.4f}   (lesson: 0.6638)")
print(f"run twice: {x2_ddim:.10f} and {x2_again:.10f} — identical, deterministic")

### Step 12 — Implement the sampler on any sub-schedule

`ddim_sample` walks any chosen list of levels $\tau_S > \dots > \tau_1$. The final hop targets level 0, where $\bar\alpha_0 = 1$ and the update collapses to $x_0 = \hat{x}_0$ — part 9's $z = 0$ rule, sharpened: the sampler ends by reporting its clean guess. First run: all 200 levels, 2,000 starting points. The `coverage` helper counts a mode as covered when at least 25 samples land within 0.5 of its center.

In [ ]:
def ddim_sample(x_start, taus):
    xt_walk = x_start.clone()
    with torch.no_grad():
        for i, t_step in enumerate(taus):
            tvec = torch.full((xt_walk.shape[0],), int(t_step))
            eps_pred = model(xt_walk, tvec)
            xhat = (xt_walk - sqrt_1m_abar[t_step - 1] * eps_pred) / sqrt_abar[t_step - 1]
            t_next = taus[i + 1] if i + 1 < len(taus) else 0
            if t_next > 0:
                xt_walk = sqrt_abar[t_next - 1] * xhat + sqrt_1m_abar[t_next - 1] * eps_pred
            else:
                xt_walk = xhat
    return xt_walk


angles_8 = 2 * math.pi * torch.arange(8) / 8
centers_8 = 4.0 * torch.stack([torch.cos(angles_8), torch.sin(angles_8)], dim=1)


def coverage(samples):
    d = torch.cdist(samples, centers_8)
    modes = int(((d < 0.5).sum(dim=0) >= 25).sum())
    near = d.min(dim=1).values.mean().item()
    radius = samples.norm(dim=1).mean().item()
    return modes, near, radius


taus_full = list(range(T, 0, -1))
x_start = torch.randn(2000, 2)
samples_200 = ddim_sample(x_start, taus_full)
modes_200, near_200, rad_200 = coverage(samples_200)

assert modes_200 >= 6
assert abs(rad_200 - 4.0) / 4.0 < 0.15

print(f"DDIM S = 200: modes covered {modes_200}/8   mean dist to nearest center {near_200:.3f}   mean radius {rad_200:.2f} (target 4.0)")

## One knob and twenty steps: eta and sub-schedules

The DDIM paper's general update adds a fresh Gaussian kick of size $\sigma_t = \eta\,\sqrt{\tilde\beta_t}$: at $\eta = 1$ the kick equals the posterior spread and the update becomes DDPM's stochastic sampler; at $\eta = 0$ the kick vanishes and the update is the deterministic DDIM we just built. Only the $\eta = 0$ end tolerates big jumps for free: the DDIM update consults nothing but $\bar\alpha$ at the jump's two endpoints, so a 20-level sub-schedule is as legitimate as the full 200. DDPM's per-step coefficients, by contrast, undo exactly one dose of noise per call — run them at only 20 of 200 levels and 180 doses are never removed. The lesson's back-of-envelope predicts a stalled blob near radius 1.4 (the 20 kept steps stretch the start by only about $e^{0.1} \approx 1.1$); the trained network's steering adds a little extra outward drift, but the samples still end far inside the ring of radius 4.

### Step 13 — Race the samplers: DDIM at S = 200, 50, 20, 10 versus naive skipping at 20

Every sampler starts from the **same** 2,000 noise points, so any quality difference is the sampler's alone. Watch the panels: DDIM barely degrades down to 20 steps and only frays at 10, while naive-skip DDPM at 20 steps never leaves the blob.

In [ ]:
def naive_skip_ddpm(x_start, n_steps):
    taus = np.linspace(T, 1, n_steps).round().astype(int)
    xt_walk = x_start.clone()
    with torch.no_grad():
        for t_step in taus:
            tvec = torch.full((xt_walk.shape[0],), int(t_step))
            eps_pred = model(xt_walk, tvec)
            coef = betas[t_step - 1] / sqrt_1m_abar[t_step - 1]
            mean = (xt_walk - coef * eps_pred) / torch.sqrt(alphas[t_step - 1])
            if t_step > 1:
                z = torch.randn_like(xt_walk)
                xt_walk = mean + torch.sqrt(betas[t_step - 1]) * z
            else:
                xt_walk = mean
    return xt_walk


runs = {"DDIM S=200": samples_200}
for S in [50, 20, 10]:
    taus_S = np.linspace(T, 1, S).round().astype(int).tolist()
    runs[f"DDIM S={S}"] = ddim_sample(x_start, taus_S)
runs["naive skip 20"] = naive_skip_ddpm(x_start, 20)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, (name, pts) in zip(axes, runs.items()):
    ax.scatter(pts[:, 0].numpy(), pts[:, 1].numpy(), s=2, alpha=0.4, color="#4ea1ff" if "DDIM" in name else "#ff7b72")
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
axes[0].set_ylabel("second coordinate")
axes[2].set_xlabel("first coordinate")
fig.suptitle("Twenty good steps beat twenty naive ones")
plt.tight_layout()
plt.show()

for name, pts in runs.items():
    m_cov, nd, r = coverage(pts)
    print(f"{name:14s}  modes covered {m_cov}/8   mean dist to center {nd:.3f}   mean radius {r:.2f}")

### Step 14 — Score the race

The asserts pin the lesson's claims: DDIM at 20 steps covers at least as many modes as naive skipping at 20 (and at least 6 of 8 outright), its mean radius stays close to the data's 4.0, and the naive samples stall at about half the data's radius — 180 doses of noise are never removed, so no amount of network steering gets them to the ring.

In [ ]:
modes_d20, near_d20, rad_d20 = coverage(runs["DDIM S=20"])
modes_n20, near_n20, rad_n20 = coverage(runs["naive skip 20"])

assert modes_d20 >= 6
assert modes_d20 >= modes_n20
assert abs(rad_d20 - 4.0) / 4.0 < 0.12
assert rad_n20 < 2.5

print(f"DDIM  S=20: modes {modes_d20}/8   mean radius {rad_d20:.2f} (data: 4.0)")
print(f"naive S=20: modes {modes_n20}/8   mean radius {rad_n20:.2f} (back-of-envelope: 1.4 before network drift; the ring sits at 4.0)")

## Determinism pays: a latent space for free

Delete the $z$ and the sampler becomes a fixed function $x_0 = F(x_T)$: same starting noise in, same sample out, bit for bit. DDPM's fresh per-step kicks make reruns from the same start diverge. We check both claims, then watch thirty codes flow along smooth trajectories onto the ring.

### Step 15 — Same code in, same sample out — bit for bit

Run DDIM twice on the same 500 starting points. `torch.equal` demands exact equality of every float — not closeness, equality. Then run the full stochastic DDPM sampler twice from the same start: the per-step kicks send paired samples whole clusters apart.

In [ ]:
codes = torch.randn(500, 2)
taus_20 = np.linspace(T, 1, 20).round().astype(int).tolist()

out_a = ddim_sample(codes, taus_20)
out_b = ddim_sample(codes, taus_20)


def ddpm_sample(x_start):
    xt_walk = x_start.clone()
    with torch.no_grad():
        for t_step in range(T, 0, -1):
            tvec = torch.full((xt_walk.shape[0],), t_step)
            eps_pred = model(xt_walk, tvec)
            coef = betas[t_step - 1] / sqrt_1m_abar[t_step - 1]
            mean = (xt_walk - coef * eps_pred) / torch.sqrt(alphas[t_step - 1])
            if t_step > 1:
                xt_walk = mean + torch.sqrt(betas[t_step - 1]) * torch.randn_like(xt_walk)
            else:
                xt_walk = mean
    return xt_walk


ddpm_a = ddpm_sample(codes)
ddpm_b = ddpm_sample(codes)
ddim_gap = (out_a - out_b).abs().max().item()
ddpm_gap = (ddpm_a - ddpm_b).norm(dim=1).mean().item()

assert torch.equal(out_a, out_b)
assert ddpm_gap > 0.5

print(f"DDIM rerun, largest coordinate difference: {ddim_gap:.1e} (exactly zero — bit for bit)")
print(f"DDPM rerun, mean distance between paired samples: {ddpm_gap:.2f} — whole clusters apart")

### Step 16 — Thirty codes flow along smooth trajectories

We record every intermediate position of 30 DDIM samples ($S = 20$) and draw each one's path from its starting code (gray dot) to its final sample (red dot). The paths bend smoothly outward and do not jitter — each endpoint is decided entirely by where its code started, which is what makes $x_T$ a meaningful latent code.

In [ ]:
traj_codes = torch.randn(30, 2)
positions = [traj_codes.clone()]
xt_traj = traj_codes.clone()

with torch.no_grad():
    for i, t_step in enumerate(taus_20):
        tvec = torch.full((30,), int(t_step))
        eps_pred = model(xt_traj, tvec)
        xhat = (xt_traj - sqrt_1m_abar[t_step - 1] * eps_pred) / sqrt_abar[t_step - 1]
        t_next = taus_20[i + 1] if i + 1 < len(taus_20) else 0
        if t_next > 0:
            xt_traj = sqrt_abar[t_next - 1] * xhat + sqrt_1m_abar[t_next - 1] * eps_pred
        else:
            xt_traj = xhat
        positions.append(xt_traj.clone())

path = torch.stack(positions).numpy()

plt.figure(figsize=(6, 6))
for j in range(30):
    plt.plot(path[:, j, 0], path[:, j, 1], lw=0.8, alpha=0.6, color="#4ea1ff")
plt.scatter(path[0, :, 0], path[0, :, 1], s=12, color="gray", label="starting code x_T")
plt.scatter(path[-1, :, 0], path[-1, :, 1], s=16, color="#ff7b72", label="final sample x_0")
plt.title("Thirty latent codes flowing onto the ring (DDIM, S = 20)")
plt.xlabel("first coordinate")
plt.ylabel("second coordinate")
plt.legend()
plt.axis("equal")
plt.show()

assert path.shape == (21, 30, 2)

print(f"trajectory array shape: {path.shape} — 21 positions (start + 20 jumps), 30 codes, 2 coordinates")

## Practice

The same problems as the lesson's practice list — the four computational ones in code, the two reasoning ones with model answers. Try each in the empty cell, then reveal the worked solution.

**Problem 1.** Score of a Gaussian. Compute the score of $\mathcal{N}(-2,\ 9)$ at $x = 1$ and at $x = -2$, read each answer as a direction and a steepness, and confirm both with a finite-difference slope of the log density.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

```python
mu_p = -2.0
sigma2_p = 9.0


def log_pdf_p(x):
    return -0.5 * (x - mu_p) ** 2 / sigma2_p - 0.5 * np.log(2 * np.pi * sigma2_p)


h_p = 1e-5
for x_eval in [1.0, -2.0]:
    formula = (mu_p - x_eval) / sigma2_p
    fd_p = (log_pdf_p(x_eval + h_p) - log_pdf_p(x_eval - h_p)) / (2 * h_p)
    print(f"x = {x_eval:+.0f}: formula {formula:+.4f}   finite difference {fd_p:+.4f}")
```

- At $x = 1$: $\frac{-2 - 1}{9} = -\frac{1}{3} \approx -0.333$. Negative, so uphill is to the **left** — from $1$ back toward the center $-2$ — with steepness one third (the distance 3 shrunk by the variance 9).
- At $x = -2$: $\frac{-2 - (-2)}{9} = 0$. You are standing on the peak; the top of the hill is flat, so there is no uphill direction to report.

**Answer:** score at $x = 1$ is $-1/3$; score at $x = -2$ is $0$. The finite differences agree to 4 decimals.

</details>

**Problem 2.** Tweedie on fresh numbers. A two-point toy has clean values $x_0 = +1$ and $x_0 = -1$ with equal odds. At some noise level $\bar\alpha_t = 0.5$, and you observe $x_t = 0.6$. Compute the posterior mean (the tanh formula with clean values $\pm c$ is $c\tanh(c\,a\,x_t/s^2)$), the score via Tweedie, and the best noise guess — then verify the bridge $\text{score} = -\mathbb{E}[\epsilon \mid x_t]/s$.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

```python
ab_p = 0.5
x_p = 0.6
a_p = np.sqrt(ab_p)
v_p = 1.0 - ab_p
s_p = np.sqrt(v_p)

post_p = 1.0 * np.tanh(1.0 * a_p * x_p / v_p)
score_p = (a_p * post_p - x_p) / v_p
eps_p = (x_p - a_p * post_p) / s_p

print(f"E[x0 | x_t] = {post_p:.4f}")
print(f"score via Tweedie = {score_p:.4f}")
print(f"best noise guess = {eps_p:.4f}")
print(f"bridge -eps/s = {-eps_p / s_p:.4f}")
```

- Shorthands: $a = \sqrt{0.5} = 0.7071$, $s^2 = 0.5$, $s = 0.7071$.
- Tanh argument: $\frac{1 \times 0.7071 \times 0.6}{0.5} = 0.8485$, so $\mathbb{E}[x_0 \mid x_t] = \tanh(0.8485) = 0.6903$.
- Tweedie: score $= \frac{0.7071 \times 0.6903 - 0.6}{0.5} = \frac{-0.1119}{0.5} = -0.2238$.
- Best noise guess: $\frac{0.6 - 0.4881}{0.7071} = 0.1582$. Bridge: $-0.1582/0.7071 = -0.2238$ — the same number.

**Answer:** $\mathbb{E}[x_0 \mid x_t] = 0.6903$, score $= -0.2238$, best noise guess $= 0.1582$. The posterior leans toward $+1$, but $x_t = 0.6$ sits slightly to the right of the shrunken posterior mean $0.4881$, so uphill points slightly left.

</details>

**Problem 3.** The denoised estimate. In part 9's worked example the model saw $x_2 = 1.9616$ at $t = 2$ (running schedule, $\bar\alpha_2 = 0.72$) and guessed $\epsilon_\theta = 0.3$. Compute $\hat{x}_0$.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

```python
x2_p = 1.9616
eps_guess = 0.3
ab2_p = 0.72

xhat_p = (x2_p - np.sqrt(1.0 - ab2_p) * eps_guess) / np.sqrt(ab2_p)

print(f"xhat0 = {xhat_p:.4f}")
```

- Constants: $\sqrt{1 - 0.72} = \sqrt{0.28} = 0.5292$ and $\sqrt{0.72} = 0.8485$.
- Numerator: $1.9616 - 0.5292 \times 0.3 = 1.8029$ — subtract the guessed noise scaled the way noise was added.
- Divide by the signal scale: $1.8029 / 0.8485 = 2.1247$.

**Answer:** $\hat{x}_0 = 2.1247$. The true $x_0$ was 2; a noise guess $0.2$ too low reads as a clean guess about $0.125$ too high.

</details>

**Problem 4.** One DDIM update by hand. Continue problem 3: at level $t = 2$ you hold $x_2 = 1.9616$, $\epsilon_\theta = 0.3$, $\hat{x}_0 = 2.1247$, and the running schedule gives $\bar\alpha_1 = 0.9$. Compute the DDIM jump to level 1, and check that rerunning it changes nothing.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

```python
ab1_p = 0.9

x1_p = np.sqrt(ab1_p) * xhat_p + np.sqrt(1.0 - ab1_p) * eps_guess
x1_rerun = np.sqrt(ab1_p) * xhat_p + np.sqrt(1.0 - ab1_p) * eps_guess

print(f"x1 = {x1_p:.4f}   rerun: {x1_rerun:.4f}")
```

- Constants: $\sqrt{0.9} = 0.9487$ and $\sqrt{0.1} = 0.3162$ — the two endpoint quantities are all the update needs.
- Signal part: $0.9487 \times 2.1247 = 2.0156$. Noise part: $0.3162 \times 0.3 = 0.0949$.
- Add: $x_1 = 2.0156 + 0.0949 = 2.1105$.

**Answer:** $x_1 = 2.1105$, every time — no $z$ enters, so the rerun agrees to every decimal. Compare part 9's Algorithm 2 trace, where a drawn $z$ entered the same step.

</details>

**Problem 5.** Reasoning with marginals. Why does a 20-step sub-schedule work for DDIM but wreck DDPM when DDPM keeps its per-step coefficients? Answer in your own words first — the criterion to reason from is: at every visited level, do the sampler's outputs still follow that level's marginal?

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- **The standard.** A sampler is on track if at every visited level $\tau$ its population of points follows the level-$\tau$ marginal — the distribution of genuinely noised data. The network trained only on such inputs (Algorithm 1 builds them with the one-line sampler); off-marginal inputs are jobs it never learned.
- **DDIM passes at any gap.** In the oracle case $\hat{x}_0 = x_0$ and the update outputs $\sqrt{\bar\alpha_{\tau'}}\,x_0 + \sqrt{1-\bar\alpha_{\tau'}}\,\epsilon$ — a legitimate draw from $q(x_{\tau'} \mid x_0)$ for **any** earlier level $\tau'$, adjacent or not: the update only consults $\bar\alpha$ at the jump's two endpoints, and the oracle algebra never used adjacency.
- **DDPM's update is a single-step tool.** Its coefficients $1/\sqrt{\alpha_t}$, $\beta_t/\sqrt{1-\bar\alpha_t}$, $\sqrt{\beta_t}$ undo exactly one dose of shrink-and-noise — that is how parts 7-9 derived it, as a Bayes flip between *adjacent* levels.
- **So skipping leaves noise in place.** Running the per-step update at 20 of 200 levels undoes 20 doses and never touches the other 180: the samples stay a blob near radius 1.4 (the 20 kept steps stretch by only $\approx e^{0.1} \approx 1.1$) instead of reaching the ring at radius 4 — exactly what Step 13 measured.
- **And the errors compound.** After the first jump, naive-skip DDPM's samples have left the marginals, so every later network call runs on inputs from the wrong distribution; DDIM's samples stay on-marginal at every visited level.

**Answer:** DDIM's update depends only on the endpoint survival fractions and lands exactly on the target marginal for jumps of any size, so 20 well-chosen levels suffice. DDPM's per-step update removes one dose per call; skipping leaves 180 doses unremoved, the samples exit the marginals immediately, and the network answers questions it was never trained on.

</details>

**Problem 6.** The latent-space consequence. Your colleague generates a striking sample with DDIM ($S = 20$) and wants to (a) reproduce it exactly next week, (b) produce ten variations that morph smoothly from it to a second sample, and (c) argue the model has not memorized the training set by showing the sample changes smoothly as its origin changes. What single object must be saved, and why does each task work with DDIM but fail with DDPM sampling? (Then, if you like, decode a few interpolated codes with `ddim_sample` and watch the morph.)

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

Save the **starting noise $x_T$** (plus the fixed sampler settings: checkpoint, $S$, sub-schedule). With DDIM, $x_0 = F(x_T)$ is a deterministic function, so $x_T$ is a complete address for the sample.

- **(a) Reproduction:** rerun $F$ on the saved code — Step 15's `torch.equal` check is exactly this guarantee.
- **(b) Variations:** decode points along the path between the two codes; $F$ is built from a smooth network and smooth arithmetic, so nearby codes give nearby samples and the outputs morph gradually.
- **(c) Smoothness as evidence:** the same experiment read as an argument — outputs vary continuously with the code, which a lookup table of memorized samples would not do.
- **Why DDPM fails all three:** Algorithm 2 draws a fresh $z$ at every step, so the output depends on $T$ unrecorded coin flips beyond $x_T$ — the map from start to sample is not a function, reruns diverge (Step 15 measured whole-cluster gaps), and any path in start-noise space is scrambled by new randomness at every decode.

```python
code_a = torch.randn(1, 2)
code_b = torch.randn(1, 2)

for lam in [0.0, 0.25, 0.5, 0.75, 1.0]:
    mixed = (1 - lam) * code_a + lam * code_b
    decoded = ddim_sample(mixed, taus_20)
    print(f"lambda = {lam:.2f}   sample = ({decoded[0, 0].item():+.3f}, {decoded[0, 1].item():+.3f})")
```

The decoded samples drift gradually from the first sample toward the second — a walk in code space is a morph in sample space.

</details>

## Wrap-up

You verified the Gaussian score formula against raw finite-difference slopes, confirmed Tweedie's formula and the score-noise bridge by three independent routes (including the lesson's hand numbers to four decimals), and proved on the toy that the denoised estimate $\hat{x}_0$ is exactly the posterior mean. The trained network's compass field visibly sharpened as $t$ fell, one DDIM jump reproduced the hand example, a 20-step sub-schedule matched the full 200 while naive skipping stalled at radius 1.4, and a bit-for-bit equality check made determinism concrete — the starting noise is now a latent code. Part 12 adds the steering wheel: guidance tilts this same score toward a chosen class with a Bayes term and a knob $w$, and latent diffusion shrinks the canvas until the whole machine fits real images.